# Encontro 5 · O mesmo eval numa plataforma: Langfuse

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-05/03-langfuse.ipynb)

Nos notebooks 01 e 02, a avaliação e os traces foram construídos à mão, para que vocês vissem o mecanismo: amostras, régua, juiz, intervalo, spans, SLOs. Em produção, ninguém mantém isso num notebook. Os times usam uma **plataforma de observabilidade e avaliação de LLM**, que guarda os traces, roda os experimentos sobre um dataset, compara versões lado a lado e bloqueia um PR quando a qualidade cai.

Este notebook leva **o mesmo agente, o mesmo golden set e as mesmas gravações** para o [Langfuse](https://langfuse.com), uma plataforma open-source que pode ser usada na nuvem ou instalada na infraestrutura da empresa.

### O que você vai sair sabendo

- Instrumentar um atendimento como trace no Langfuse: um span de agente, uma *generation* por chamada ao modelo, um span de ferramenta por busca, com tokens, custo e duração.
- Subir o golden set como **dataset** e rodar um **experimento** por versão, com a régua e o juiz como avaliadores.
- Escrever o **gate de regressão** que a plataforma roda num pull request.
- Dizer o que a plataforma faz por vocês e o que continua sendo decisão de vocês.

### Mapa do notebook

| Parte | O que você faz | Onde olhar no Langfuse |
|---|---|---|
| 0 · Ambiente e chaves | — | Settings → API Keys |
| 1 · Um atendimento como trace | envia o atendimento mais lento da v1 | Tracing → Traces |
| 2 · O golden set como dataset | sobe os 25 casos | Datasets |
| 3 · Um experimento por versão | v1 e v2, com régua e juiz | Datasets → aurora-golden-e5 → Experiments |
| 4 · O gate de regressão | a regra que bloqueia o PR | — |
| 5 · Síntese | o que a plataforma fez, o que continuou seu | — |

### O que é preciso

**Nenhuma chave de modelo.** Os atendimentos vêm da gravação, como nos notebooks 01 e 02. É preciso só uma conta no Langfuse, que tem plano gratuito:

1. Crie uma conta em [cloud.langfuse.com](https://cloud.langfuse.com) e um projeto (por exemplo, `aurora`).
2. Em **Settings → API Keys**, crie um par de chaves.
3. No Colab, abra **Segredos** (o ícone de chave, na barra lateral) e crie `LANGFUSE_PUBLIC_KEY` e `LANGFUSE_SECRET_KEY`. Se a conta foi criada na região dos EUA, crie também `LANGFUSE_BASE_URL` com `https://us.cloud.langfuse.com`.

Nunca cole uma chave numa célula: o notebook pode ser compartilhado com ela dentro.

## Parte 0 · Ambiente e chaves

In [ ]:
# Etapa 0 · No Colab, baixa o repositório da disciplina. Na sua máquina, não faz nada.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-05")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "langfuse>=5,<6"], check=True)
sys.path.insert(0, os.getcwd())

import avaliacao as av
av.preparar("usar")
golden = av.carregar_golden()
v1, v2 = av.rodar("v1", golden), av.rodar("v2", golden)
av.julgar(v1, "evidencia"); av.julgar(v2, "evidencia")
print("Gravações prontas.")

In [ ]:
# As chaves vêm dos Segredos do Colab (ou das variáveis de ambiente, na sua máquina).
for nome in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"):
    if not os.environ.get(nome) and "google.colab" in sys.modules:
        from google.colab import userdata
        try:
            os.environ[nome] = userdata.get(nome)
        except Exception:
            pass

from langfuse import Langfuse
lf = Langfuse()
print("Conectado ao Langfuse." if lf.auth_check() else "As chaves não foram aceitas: confira os Segredos e a região.")

## Parte 1 · Um atendimento como trace

**Pergunta:** como o trace que vimos na cascata do notebook 01 aparece numa plataforma?

A função abaixo pega um atendimento já rodado (da gravação) e o recria no Langfuse, passo a passo. Cada evento do trace da Aula 4 vira uma **observação**:

| Evento do trace | Observação no Langfuse | O que carrega |
|---|---|---|
| o atendimento | `agent` (a raiz) | a pergunta, a resposta, a versão |
| uma chamada ao modelo | `generation` | o modelo, os tokens, o custo e a duração da gravação |
| uma busca | `tool` | os argumentos e as páginas devolvidas |

Num projeto real, essas observações seriam criadas **enquanto** o agente roda, com o mesmo `start_as_current_observation` em volta de cada chamada. Aqui, como o atendimento já está gravado, ela os recria depois. Com `ritmo=1.0`, a função espera o tempo de cada chamada, para que a linha do tempo no Langfuse fique igual à da gravação.

In [ ]:
import json, time
from langfuse import propagate_attributes


def _precos(rodada):
    cab = av._cabecalho(av.GRAVACOES[av.VERSOES[rodada["versao"]]["gravacao"]])
    return cab.get("modelo"), cab.get("preco_entrada", 0), cab.get("preco_saida", 0)


def instrumentar(linha, rodada, ritmo=0.0):
    """Cria, dentro da observação atual, uma generation por chamada ao modelo e um tool por busca."""
    nome_modelo, p_ent, p_sai = _precos(rodada)
    passo = 0
    for e in linha["trace"]:
        if e["evento"] == "modelo":
            passo += 1
            seg = e.get("segundos", 0.0)
            g = lf.start_observation(
                as_type="generation", name=f"modelo · passo {passo}", model=nome_modelo,
                usage_details={"input": e["tokens_entrada"], "output": e["tokens_saida"]},
                cost_details={"input": e["tokens_entrada"] * p_ent / 1e6, "output": e["tokens_saida"] * p_sai / 1e6},
                output={"pediu": e["detalhe"]})
            time.sleep(seg * ritmo)
            g.end(end_time=time.time_ns() + int(seg * (1 - ritmo) * 1e9))    # a duração é a da gravação
        elif e["evento"] == "ferramenta":
            nome, args = e["detalhe"].split("(", 1)
            t = lf.start_observation(as_type="tool", name=nome, input=json.loads(args[:-1]),
                                     output={"paginas": e["devolveu"]})
            t.end()


def enviar_atendimento(rodada, caso_id, amostra=0, ritmo=1.0):
    """Envia um atendimento gravado como um trace, com a régua e o juiz como scores."""
    l = av._linha(rodada, caso_id, amostra)
    with propagate_attributes(trace_name=f"atendimento-{caso_id}", version=rodada["versao"],
                              tags=[rodada["versao"], "encontro-05"],
                              metadata={"caso": caso_id, "amostra": str(amostra)}):
        with lf.start_as_current_observation(as_type="agent", name="atendimento",
                                             input={"pergunta": l["pergunta"]}) as raiz:
            instrumentar(l, rodada, ritmo)
            raiz.update(output={"desfecho": l["desfecho"], "fontes": l["fontes"], "resposta": l["resposta"]})
            raiz.score_trace(name="regua", value=1.0 if l["acertou"] else 0.0, data_type="BOOLEAN",
                             comment=l["motivo"])
            if "juiz_evidencia" in l:
                raiz.score_trace(name="juiz_evidencia", value=1.0 if l["juiz_evidencia"]["aprovado"] else 0.0,
                                 data_type="BOOLEAN", comment=l["juiz_evidencia"]["justificativa"][:500])
    lf.flush()
    return raiz.trace_id

Agora, o mesmo atendimento que abrimos na cascata do notebook 01: o mais lento da v1. Com `ritmo=1.0`, a célula leva o tempo do atendimento original.

In [ ]:
LENTO, AMOSTRA = av.mais_lento(v1)
trace_id = enviar_atendimento(v1, LENTO, AMOSTRA, ritmo=1.0)
print(f"{LENTO} (amostra {AMOSTRA}) enviado. Abra no Langfuse:")
print(lf.get_trace_url(trace_id=trace_id))

**O que olhar no Langfuse.**

- A **linha do tempo** do trace é a cascata do notebook 01. A plataforma desenha sozinha.
- Clique numa *generation*: o modelo, os tokens e o **custo**, calculado com o preço que nós informamos. Sem o preço, o Langfuse usa a tabela de preços dele, quando conhece o modelo.
- Na aba de **scores**, a régua e o juiz, com o motivo e a justificativa como comentário. Um score pode vir de código, de um juiz ou de uma pessoa clicando 👍 na interface: é assim que a anotação humana da Parte 2 do notebook 01 vira dado.
- As **tags** e a **versão** permitem filtrar: todos os traces da v2, todos os atendimentos com régua 0.

**Pergunta para a turma.** Este trace tem a pergunta do colaborador e a resposta. Em produção, isso é dado pessoal? Quanto tempo o trace pode ficar guardado, e quem pode lê-lo? A pergunta volta no Encontro 6, com a LGPD.

## Parte 2 · O golden set como dataset

**Pergunta:** onde o golden set mora, para que o time inteiro use o mesmo?

No Langfuse, um **dataset** guarda os casos com a entrada e a saída esperada. Ele tem versão: quando alguém adiciona um caso, os experimentos antigos continuam ligados à versão em que rodaram. A célula abaixo sobe os 25 casos; rodar de novo não duplica nada, porque cada item tem um id fixo.

In [ ]:
DATASET = "aurora-golden-e5"
try:
    lf.create_dataset(name=DATASET, description="Golden set do Encontro 5: 19 casos dos Encontros 1 a 4 e 6 da produção.")
except Exception:
    pass                                        # o dataset já existe
for c in golden:
    lf.create_dataset_item(dataset_name=DATASET, id=f"{DATASET}-{c['id']}",
                           input={"pergunta": c["pergunta"]},
                           expected_output={k: c[k] for k in ("esperado", "resposta_contem_algum", "resposta_nao_contem") if k in c},
                           metadata={"caso": c["id"], "categoria": c["categoria"], "origem": c["origem"]})
dataset = lf.get_dataset(DATASET)
print(f"{len(dataset.items)} itens no dataset {DATASET}")

## Parte 3 · Um experimento por versão

**Pergunta:** como comparar a v1 e a v2 na plataforma?

Um **experimento** roda uma tarefa em cada item do dataset, passa o resultado pelos avaliadores e guarda tudo junto: os traces, os scores de cada item e os scores do experimento inteiro. Três peças:

| Peça | O que é aqui |
|---|---|
| **tarefa** | atende o caso. Aqui, reproduz o atendimento gravado; num projeto real, chamaria o agente |
| **avaliadores de item** | a régua (código) e o juiz com evidência (modelo), um score por item |
| **avaliadores do experimento** | o acerto médio e o custo por atendimento, um score para o experimento |

In [ ]:
from langfuse import Evaluation
from comum.resultado import Resultado

RODADAS = {"v1": v1, "v2": v2}
POR_ID = {c["id"]: c for c in golden}


def tarefa_da(versao, amostra=0):
    rodada = RODADAS[versao]

    def tarefa(*, item, **kwargs):
        l = av._linha(rodada, item.metadata["caso"], amostra)
        instrumentar(l, rodada, ritmo=0.0)
        return {"desfecho": l["desfecho"], "fontes": l["fontes"], "resposta": l["resposta"],
                "custo_usd": l["custo_usd"], "juiz": l.get("juiz_evidencia", {})}
    return tarefa


def regua(*, output, metadata, **kwargs):
    """A régua dos Encontros 1 a 4, como avaliador: desfecho, fontes e valor."""
    ok, motivo = av.ma.avaliar(Resultado.de_dict(output), POR_ID[metadata["caso"]])
    return Evaluation(name="regua", value=1.0 if ok else 0.0, comment=motivo, data_type="BOOLEAN")


def juiz(*, output, **kwargs):
    """O juiz com evidência. O veredito vem da gravação, como o resto do atendimento."""
    v = output["juiz"]
    return Evaluation(name="juiz_evidencia", value=1.0 if v.get("aprovado") else 0.0,
                      comment=v.get("justificativa", "")[:500], data_type="BOOLEAN")


def acerto_medio(*, item_results, **kwargs):
    notas = [e.value for r in item_results for e in r.evaluations if e.name == "regua"]
    return Evaluation(name="acerto_medio", value=sum(notas) / len(notas))


def custo_medio(*, item_results, **kwargs):
    custos = [r.output["custo_usd"] for r in item_results if r.output]
    return Evaluation(name="custo_por_atendimento", value=sum(custos) / len(custos))


def experimento(versao, amostra=0):
    return dataset.run_experiment(
        name=f"Aurora · {av.VERSOES[versao]['nome']}", run_name=f"{versao}-amostra-{amostra}",
        description=av.VERSOES[versao]["descricao"], task=tarefa_da(versao, amostra),
        evaluators=[regua, juiz], run_evaluators=[acerto_medio, custo_medio],
        max_concurrency=1, metadata={"versao": versao, "amostra": str(amostra)})

In [ ]:
resultado_v1 = experimento("v1")
resultado_v2 = experimento("v2")
lf.flush()
for r in (resultado_v1, resultado_v2):
    notas = {e.name: e.value for e in r.run_evaluations}
    print(f"{r.run_name:<16} acerto {notas['acerto_medio']:.0%} · US$ {notas['custo_por_atendimento']:.5f} por atendimento")
    print(f"{'':<16} {r.experiment_url}")

**O que olhar no Langfuse.** Em **Datasets → aurora-golden-e5**, os dois experimentos aparecem lado a lado: o acerto e o custo de cada um, e a tabela item por item, com o trace de cada atendimento a um clique. É o `mapa_de_casos` do notebook 01, mantido pela plataforma.

**E o que a plataforma não fez.** Cada experimento é **uma amostra**: uma rodada da suíte. O Langfuse mostra a diferença entre as duas rodadas, mas não diz se ela é maior que o ruído. Para isso, o time roda o mesmo experimento várias vezes (`experimento("v1", amostra=1)`, e assim por diante) e compara, como no teste A/A. **A plataforma guarda e mostra; a estatística continua sendo do time.**

## Parte 4 · O gate de regressão

**Pergunta:** como isso bloqueia um pull request?

O Langfuse tem uma GitHub Action, a [`langfuse/experiment-action`](https://github.com/langfuse/experiment-action), que roda um script de experimento a cada PR, comenta o resultado no PR e **falha o check** quando o script levanta um `RegressionError`. O script só precisa de uma função chamada `experiment`, que recebe um contexto com o dataset já carregado:

```python
from langfuse import RegressionError, RunnerContext

def experiment(context: RunnerContext):
    resultado = context.run_experiment(
        name="Aurora · PR", task=tarefa_do_agente,
        evaluators=[regua, juiz], run_evaluators=[acerto_medio, custo_medio],
    )
    acerto = next(e.value for e in resultado.run_evaluations if e.name == "acerto_medio")
    if acerto < 0.80:
        raise RegressionError(result=resultado, metric="acerto_medio", value=acerto, threshold=0.80)
    return resultado
```

No workflow do GitHub, o passo do action recebe as chaves do Langfuse (como segredos do repositório), o caminho do script e o nome do dataset. A versão do action e os parâmetros estão no README do action.

A célula abaixo aplica a mesma regra aos dois experimentos da Parte 3:

In [ ]:
from langfuse import RegressionError

LIMITE = 0.80
for r in (resultado_v1, resultado_v2):
    acerto = next(e.value for e in r.run_evaluations if e.name == "acerto_medio")
    try:
        if acerto < LIMITE:
            raise RegressionError(result=r, metric="acerto_medio", value=acerto, threshold=LIMITE)
        print(f"{r.run_name:<16} passa no gate ({acerto:.0%} ≥ {LIMITE:.0%})")
    except RegressionError:
        print(f"{r.run_name:<16} BLOQUEADO: acerto {acerto:.0%} < {LIMITE:.0%}")

**O que esse gate ainda não faz, e vocês sabem fazer.** Ele compara uma rodada com um número fixo. Com o que vocês viram no notebook 01, um gate melhor:

1. roda **várias amostras** e compara o intervalo da diferença com a versão em produção, e não um número com um limite;
2. tem uma regra separada para os **casos críticos** (`X1`, `G6`, os sem resposta), porque a média esconde;
3. olha o **custo por acerto** junto com o acerto, para não aprovar uma versão que acerta o mesmo e custa o dobro.

Tudo isso cabe dentro da função `experiment`: é código Python, e a régua, o juiz e o bootstrap do `avaliacao.py` podem ir para lá.

## Parte 5 · Síntese

### O que a plataforma fez por vocês, e o que continuou seu

| A plataforma faz | Continua sendo decisão do time |
|---|---|
| guarda os traces, com tokens, custo e tempo | o que entra no trace, e por quanto tempo (LGPD) |
| guarda o golden set, com versão | quais casos entram, e o gabarito de cada um |
| roda o experimento e mostra os scores lado a lado | quantas amostras rodar, e se a diferença é ruído |
| chama o juiz e guarda o veredito | a rubrica do juiz, e a calibração contra humanos |
| falha o PR quando o script manda | a regra do gate: o limite, os casos críticos, o custo |

### Langfuse, LangSmith, Phoenix

| | Langfuse | LangSmith | Arize Phoenix |
|---|---|---|---|
| Modelo | open-source; nuvem ou instalado na empresa | produto da LangChain; nuvem (e instalação na empresa nos planos enterprise) | open-source; local ou na empresa |
| Melhor quando | controle de infraestrutura e residência de dados pesam | o stack é LangChain e LangGraph, e o time quer a integração mais profunda | o time quer começar local e já usa OpenTelemetry |
| Instrumentação | SDK próprio sobre OpenTelemetry, integrações com os frameworks | automática para LangChain e LangGraph, SDK para o resto | OpenTelemetry e OpenInference |

As três fazem traces, datasets, experimentos e juízes. A escolha segue o critério do Encontro 4: estado e dados onde a empresa precisa, profundidade de observabilidade e grau de lock-in. Como o Langfuse e o Phoenix falam OpenTelemetry, a instrumentação feita para um serve, em boa parte, para o outro.